# PnL Calculation

## 1. Install Packages

In [2]:
import sys
sys.path.append('/Users/fabioballoni/Development/repos/risk/')
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

## 2. Import Modules

In [3]:
from importlib import reload
import numpy as np
from datetime import date, datetime as _datetime, timedelta
import datetime
from risk_pylibrary import config as CF
from risk_pylibrary import pandas_patched as pd
from tools.snowflake_db import db_connection as db
from risk_pylibrary.projects.pnl import pnl_support as pnl_sup
from risk_pylibrary.projects.pnl import pnl_fifo as pnl


     ------------------------------------------------------------
               Risk @ Trade Republic Bank GmbH
     ------------------------------------------------------------
      Node: fabioballoni	User: fabioballoni
      Python version 3.12.8 (v3.12.8:2dc476bcb91, Dec  3 2024, 14:43:19) [Clang 13.0.0 (clang-1300.0.29.30)]
      pandas: 2.2.3	numpy: 2.2.3
      Kernel started 2025-03-04 23:54
     ------------------------------------------------------------


### 2.1. Import Custom PnL Modules

In [5]:
reload(pnl_sup)
reload(pnl)
reload(db)

<module 'risk_pylibrary.tools.snowflake_db.db_connection' from '/root/risk_pylibrary/tools/snowflake_db/db_connection.py'>

In [ ]:
reload(db)




## 3. PnL Framework

In [13]:
#start_list=[date(2022,11,1),date(2022,12,1),date(2023,1,1),date(2023,2,1),date(2023,3,1),date(2023,4,1),date(2023,5,1),date(2023,6,1),date(2023,7,1),date(2023,8,1),date(2023,9,1),date(2023,10,1),date(2023,11,1),date(2023,12,1)]
#end_list=[date(2022,11,30),date(2022,12,31),date(2023,1,31),date(2023,2,28),date(2023,3,31),date(2023,4,30),date(2023,5,31),date(2023,6,30),date(2023,7,31),date(2023,8,31),date(2023,9,30),date(2023,10,31),date(2023,11,30),date(2023,12,31)]
start_list=[date(2025,2,8)]
end_list=[date(2025,2,28)]
instr_types=['STOCK','BOND','FUND']
#account_list=['caligula','trajan','caracalla','tiberius']#'
account_list=['tiberius']
skip_ddate_list=True
skip_cache=False
fixed_startpoint=None#date(2024,1,2)
fixed_endpoint=None#date(2024,8,27)
counter_sd=1
counter_ed=21
append_out_put=False

In [5]:
_datetime.strftime((start_list[0] - timedelta(counter_sd)),"%Y%m%d")

'20250208'

In [14]:
_datetime.strftime((end_list[0] - timedelta(counter_ed)),"%Y%m%d")

'20250207'

In [9]:
tmp_cache=pd.read_pickle('~/Development/repos/risk/risk_pylibrary/projects/pnl/storage/20250207_20250207_trajan_all_recon_tmp_cache_total.pkl')

### 3.1. Build Cache

In [ ]:
import multiprocessing
print('Cores Available: %s'%multiprocessing.cpu_count())
n_cores = multiprocessing.cpu_count();
enable_multi_proc=False
if enable_multi_proc is True:
     print('Cores in Usage: %s'%n_cores)
#
pd.options.mode.chained_assignment = None
import warnings
warnings.filterwarnings('ignore')
CF.cache_pnl_prices={}
CF.cache_pnl_trade={}

for account in account_list:
    
    if account == 'caligula':
        query_eis=True
        print('\n \t Using EIS TRading Data for account: %s'%(account))
    else:
        query_eis=False
        
    print('\n \t Starting Iteration for %s at: %s'%(account, _datetime.now()))
    print('\n \t\t Fetching Trades for dates %s and %s'%(start_list[0] if fixed_startpoint is None else fixed_startpoint, end_list[-1] if fixed_endpoint is None else fixed_endpoint))
    
    df_trades = pnl_sup.get_trades_pnl(account=account,
                                       startdate=start_list[0] if fixed_startpoint is None else fixed_startpoint, 
                                       enddate=end_list[-1] if fixed_endpoint is None else fixed_endpoint,
                                       add_filter=None,
                                       query_eis=query_eis,
                                       instrument_type=instr_types).sort_values(by='time')
    df_trades['price']=df_trades['price'].astype(float)
    df_trades_filtered=df_trades.copy()
    df_trades_filtered=df_trades_filtered.sort_values(by='time')


    # Setting Custom 
    if start_list is None:
        print('\n \t Setting a startlist based on trades max date')
        start_list=df_trades['time'].apply(lambda x: x.date()).unique()

    if end_list is None:
        print('\n \t Setting an endlist by copying the startlist')
        end_list=start_list
        
    
    # Get Prices
    df_prx = pnl_sup.get_prices_eod(startdate=start_list[0], enddate=end_list[-1], read_full_dwh=True)

    
    # Get Cache
    if skip_cache is False:
        start_str_cache = _datetime.strftime((start_list[0] - timedelta(counter_sd)),"%Y%m%d")
        end_str_cache = _datetime.strftime((end_list[0] - timedelta(counter_ed)),"%Y%m%d")
        
        print("\n \t\t Getting Cache on Dates: %s and %s"%(start_list[0] - timedelta(counter_sd),end_list[0] - timedelta(counter_ed)))
        tmp_cache=pd.read_pickle('~/Development/repos/risk/risk_pylibrary/projects/pnl/storage/%s_%s_%s_all_recon_tmp_cache_total.pkl'%(start_str_cache,end_str_cache,account))
        
        print("\t\t\t Slicing following instrument_types: %s"%instr_types)
        tmp_cache=tmp_cache[tmp_cache.instrument_type.isin(instr_types)]
        
        # FB 20241001 Excluding JP3734800000 as it causes an error
        # no position in there but a stock split on the 27th September
        if account == 'caracalla':
            tmp_cache = tmp_cache[~tmp_cache.symbol.isin(['JP3734800000','JP3493800001','JP3709600005'])]
        #elif account == 'caligula':
        #    instr_excl = ['US67066G1040','DE000ENER6Y0','NL0010273215']
        #    tmp_cache = tmp_cache[tmp_cache.symbol.isin(instr_excl)]
        
    else:
        tmp_cache = pd.DataFrame()
    

    if append_out_put:
        out_pos=pd.DataFrame()
    
    for startdate, enddate in dict(zip(start_list,end_list)).items():
        
        # Print Start and Enddate
        print('\n \t\t Slicing Trades for dates %s and %s'%(startdate, enddate))
        # Slice Trades
        tmp_trades = df_trades_filtered[df_trades_filtered['time'] <= _datetime.combine(enddate, _datetime.max.time())]
        tmp_trades = tmp_trades[tmp_trades['time'] >= _datetime.combine(startdate, _datetime.min.time())]

        if tmp_cache.empty is False:
            print('\n \t\t Cache - Number of Row: %s'%len(tmp_cache))
            print('\t\t Cache - Number of Unique Symbols: %s'%len(tmp_cache.symbol.unique()))

            if 'account' in tmp_cache.columns:
                tmp_cache = tmp_cache.drop(['report_date', 'account'],axis=1)


        # Concat Print New Trades Cache
        total_trade_cache = pd.concat([tmp_cache,tmp_trades],axis=0)
        
        #if account == 'caligula':
        #    instr_excl = ['US67066G1040','DE000ENER6Y0','NL0010273215']
        #    print('\n \t\t Excluding the following instruments: %s'%instr_excl)
        #    total_trade_cache = total_trade_cache[total_trade_cache.symbol.isin(instr_excl)]
            
        CF.cache_pnl_trade = total_trade_cache
        print('\n \t\t Trades tmp Trades - Number of Row: %s'%len(tmp_trades))
        print('\t\t Trades Cache (tmp and cache) - Number of Row: %s'%len(CF.cache_pnl_trade))
        print('\t\t Trades Cache (tmp and cache) - Number of Unique Symbols: %s'%len(CF.cache_pnl_trade.symbol.unique()))
        
                
        # Slicing Price Cache
        print('\n \t\t Slicing Prices for dates %s and %s'%(startdate, enddate))
        
        #tmp_prx = df_prx[CF.cache_pnl_trade.symbol.unique()]
        tmp_prx=df_prx
        #print('\n \t\t Prices Cache - Number of nans: %s'%(len(tmp_prx.T)-len(tmp_prx.T.dropna())))
        
        CF.cache_pnl_prices = tmp_prx[startdate:enddate]

        
        # Count unique instrumente and ventually scale down n_cores
        len_instr = len(CF.cache_pnl_trade.symbol.unique())
        if n_cores > len_instr:
                      print("\n Scaling down n_cores to: %s"%len_instr)
                      n_cores_res = len_instr
        else:
            n_cores_res = n_cores
        
        if enable_multi_proc:
            tmp_pos,tmp_cache, tmp_trades = pnl.pnl_fifo_engine(startdate=startdate, enddate=enddate,multi_proc=True,
                                                                      n_cores = n_cores_res,read_cache_trades=True,read_price_cache=True,
                                                                      skip_ddate_list=skip_ddate_list,verbose=False)
        else:
            tmp_pos,tmp_cache, tmp_trades = pnl.pnl_fifo_engine(startdate=startdate, enddate=enddate,read_cache_trades=True,read_price_cache=True,
                                                          skip_ddate_list=skip_ddate_list,verbose=False)

        # Dump files
        start_str=str(startdate.year)+('0'+str(startdate.month) if + startdate.month<10 else str(startdate.month)) +('0'+str(startdate.day) if startdate.day<10 else str(startdate.day));
        end_str=str(startdate.year)+('0'+str(enddate.month) if + enddate.month<10 else str(enddate.month)) +('0'+str(enddate.day) if enddate.day<10 else str(enddate.day));
        # Pos Format
        tmp_pos['account']=account;
        tmp_pos=tmp_pos.dropna();
        tmp_pos=tmp_pos[tmp_pos.upnl.abs()!=np.inf];
        #
        # Cache Format
        tmp_cache['report_date']=_datetime(enddate.year,enddate.month,enddate.day,23,59,59);
        tmp_cache['account']=account;
        instr_type_str = 'total' if len(instr_types)>=2 else instr_types[0]
        
        if append_out_put:
            # Append Positions to pos
            out_pos = pd.concat([out_pos, tmp_pos], axis=0)
        else:
            tmp_pos.to_pickle('~/Development/repos/risk/risk_pylibrary/projects/pnl/storage/%s_%s_%s_all_recon_tmp_pos_%s.pkl'%(start_str,end_str,account,instr_type_str));
        
        
        tmp_cache.to_pickle('~/Development/repos/risk/risk_pylibrary/projects/pnl/storage/%s_%s_%s_all_recon_tmp_cache_%s.pkl'%(start_str,end_str,account,instr_type_str));
        #tmp_cache.to_csv('/root/risk_pylibrary/projects/pnl/storage/caligula_eis/%s_%s_%s_all_recon_tmp_cache_%s.csv'%(start_str,end_str,account,instr_type_str),index=False);
        #tmp_pos.to_csv('/root/risk_pylibrary/projects/pnl/storage/caligula_eis/%s_%s_%s_all_recon_tmp_pos_%s.csv'%(start_str,end_str,account,instr_type_str),index=False);
    
    # Saving Positions Output to file in case of summary output
    if append_out_put:
        out_pos.to_pickle('~/Development/repos/risk/risk_pylibrary/projects/pnl/storage/%s_%s_%s_all_recon_tmp_pos_%s.pkl'%(start_str,end_str,account,instr_type_str));
        

Cores Available: 8

 	 Starting Iteration for tiberius at: 2025-03-05 00:02:55.114701

 		 Fetching Trades for dates 2025-02-08 and 2025-02-28
	 Retrieving Full DWH Prices

 		 Getting Cache on Dates: 2025-02-07 and 2025-02-07
			 Slicing following instrument_types: ['STOCK', 'BOND', 'FUND']

 		 Slicing Trades for dates 2025-02-08 and 2025-02-28

 		 Cache - Number of Row: 2451
		 Cache - Number of Unique Symbols: 387

 		 Trades tmp Trades - Number of Row: 80297
		 Trades Cache (tmp and cache) - Number of Row: 82748
		 Trades Cache (tmp and cache) - Number of Unique Symbols: 390

 		 Slicing Prices for dates 2025-02-08 and 2025-02-28

 Adjusting for Corporate Actions ***************************************

 Reading Cached Trades ***************************************

 Retrieving  Prices ***************************************
	 Reading Price Cache

 Initialising PnL Calculation ***************************************
	 Calculating Realised and Unrealised PnL for 2025-02-28

	 Init

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [16]:
postm()

NameError: name 'postm' is not defined

In [9]:
out_cache=pd.read_pickle('20250128_20250128_caligula_all_recon_tmp_cache_total_BONDS_ETFS.pkl')

In [15]:
out_pos=pd.read_pickle('20250128_20250128_caligula_all_recon_tmp_pos_total_BONDS_ETFS.pkl')

In [10]:
out_cache=pd.concat([out_cache,pd.read_pickle('20250128_20250128_caligula_all_recon_tmp_cache_STOCK.pkl')],axis=0)

In [16]:
out_pos=pd.concat([out_pos,pd.read_pickle('20250128_20250128_caligula_all_recon_tmp_pos_STOCK.pkl')],axis=0)

In [18]:
len(out_pos.symbol.unique())

12162

In [22]:
out_pos.rpnl.sum()

11152.958917641516

In [14]:
out_cache.to_pickle('20250128_20250128_caligula_all_recon_tmp_cache_total.pkl')

In [19]:
out_pos.to_pickle('20250128_20250128_caligula_all_recon_tmp_pos_total.pkl')